<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Block 1 — Load 19-column final dataset**

In [3]:
# ============================================================
# ML-05 / W05 — BLOCK 1
# LOAD FINAL 19-COLUMN FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/content/final_features_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 70)
print("FINAL FEATURE DATASET")
print("=" * 70)

print("Rows    :", f"{len(df):,}")
print("Columns :", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

print("\nShape:", df.shape)

FINAL FEATURE DATASET
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape: (2871202, 19)


**Block 2 — Missing indicators + 90-day eligibility**

In [14]:
# ============================================================
# BLOCK 2
# MISSING INDICATORS + BASIC DATA QUALITY
# ============================================================

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Remove meaningless missing indicator
# ------------------------------------------------------------

DROP_INDICATOR = "ai_other_missing"

if DROP_INDICATOR in df.columns:
    df = df.drop(columns=[DROP_INDICATOR])

print("=" * 70)
print("MISSING INDICATOR DECISION")
print("=" * 70)

print("Dropped:", DROP_INDICATOR)

remaining_indicators = [
    c for c in df.columns
    if c.endswith("_missing")
]

print("\nMissing indicators kept:")
for c in remaining_indicators:
    print("KEEP ->", c)

# ------------------------------------------------------------
# 2. Sort page-wise chronologically
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Check page history
# ------------------------------------------------------------

page_history = (
    df
    .groupby("content_hash_id")["month"]
    .nunique()
)

print("\n" + "=" * 70)
print("PAGE HISTORY")
print("=" * 70)

print(
    "Total pages:",
    f"{len(page_history):,}"
)

print(
    "Pages with < 3 months:",
    f"{(page_history < 3).sum():,}"
)

print(
    "Pages with >= 3 months:",
    f"{(page_history >= 3).sum():,}"
)

# ------------------------------------------------------------
# 4. Check monthly continuity
# ------------------------------------------------------------

df["previous_month"] = (
    df
    .groupby("content_hash_id")["month"]
    .shift(1)
)

df["month_gap"] = (
    (
        df["month"].dt.year
        - df["previous_month"].dt.year
    ) * 12
    +
    (
        df["month"].dt.month
        - df["previous_month"].dt.month
    )
)

gap_rows = df[
    df["month_gap"].notna()
    &
    (df["month_gap"] > 1)
]

print("\n" + "=" * 70)
print("MONTHLY CONTINUITY")
print("=" * 70)

print(
    "Rows with a month gap:",
    f"{len(gap_rows):,}"
)

print(
    "Pages affected:",
    f"{gap_rows['content_hash_id'].nunique():,}"
)

# helper columns no longer needed
df = df.drop(
    columns=[
        "previous_month",
        "month_gap"
    ]
)

MISSING INDICATOR DECISION
Dropped: ai_other_missing

Missing indicators kept:
KEEP -> gsc_avg_position_missing
KEEP -> ga4_total_engagement_sec_missing
KEEP -> sessions_organic_missing
KEEP -> sessions_ai_missing

PAGE HISTORY
Total pages: 427,292
Pages with < 3 months: 47,141
Pages with >= 3 months: 380,151

MONTHLY CONTINUITY
Rows with a month gap: 13,014
Pages affected: 11,525


**Block 3 — Feature impact + leakage checks**

In [15]:
# ============================================================
# BLOCK 3
# FEATURE AUDIT + LEAKAGE + VIF
# ============================================================

from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# 1. ID columns are NOT model features
# ------------------------------------------------------------

ID_COLUMNS = [
    "client_hash_id",
    "content_hash_id"
]

# ------------------------------------------------------------
# 2. Potential future/label-derived names
# ------------------------------------------------------------

future_keywords = [
    "future",
    "target",
    "label",
    "decay",
    "next_",
    "lead_"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in future_keywords
    ):
        suspicious_columns.append(col)

print("=" * 70)
print("LEAKAGE NAME CHECK")
print("=" * 70)

if suspicious_columns:
    print("Potentially suspicious columns:")
    for col in suspicious_columns:
        print("CHECK ->", col)
else:
    print("PASS: No obvious future/target-derived columns.")

# ------------------------------------------------------------
# 3. Constant columns
# ------------------------------------------------------------

feature_candidates = [
    c for c in df.columns
    if c not in ID_COLUMNS + ["month"]
]

constant_columns = [
    c for c in feature_candidates
    if df[c].nunique(dropna=False) <= 1
]

print("\n" + "=" * 70)
print("ZERO-VARIANCE CHECK")
print("=" * 70)

if constant_columns:
    for c in constant_columns:
        print("DROP ->", c)
else:
    print("PASS: No zero-variance columns.")

# ------------------------------------------------------------
# 4. Numeric feature list
# ------------------------------------------------------------

numeric_features = [
    c for c in feature_candidates
    if c not in constant_columns
    and pd.api.types.is_numeric_dtype(df[c])
]

print("\nNumeric model candidates:")
for c in numeric_features:
    print(" -", c)

# ------------------------------------------------------------
# 5. Correlation between current features
# ------------------------------------------------------------

corr_matrix = (
    df[numeric_features]
    .corr()
)

print("\n" + "=" * 70)
print("HIGH FEATURE-TO-FEATURE CORRELATION")
print("=" * 70)

high_corr_pairs = []

for i in range(len(numeric_features)):

    for j in range(i + 1, len(numeric_features)):

        a = numeric_features[i]
        b = numeric_features[j]

        corr = corr_matrix.loc[a, b]

        if abs(corr) >= 0.90:

            high_corr_pairs.append(
                (a, b, round(corr, 3))
            )

if high_corr_pairs:

    for pair in high_corr_pairs:
        print(pair)

else:

    print("No feature pairs with |correlation| >= 0.90")

# ------------------------------------------------------------
# 6. VIF sample
# ------------------------------------------------------------
# VIF on millions of rows is unnecessarily expensive.
# We use a reproducible sample for diagnostic purposes.

vif_sample = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sample(
        n=min(100_000, len(df)),
        random_state=42
    )
)

# Remove columns with zero variance inside sample
vif_features = [
    c for c in numeric_features
    if vif_sample[c].nunique() > 1
]

X_vif = vif_sample[vif_features]

vif_table = pd.DataFrame({
    "feature": vif_features,
    "VIF": [
        variance_inflation_factor(
            X_vif.values,
            i
        )
        for i in range(X_vif.shape[1])
    ]
})

vif_table = (
    vif_table
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("VIF TEST")
print("=" * 70)

display(vif_table)

LEAKAGE NAME CHECK
PASS: No obvious future/target-derived columns.

ZERO-VARIANCE CHECK
PASS: No zero-variance columns.

Numeric model candidates:
 - gsc_clicks
 - gsc_impressions
 - gsc_avg_position
 - ga4_total_engagement_sec
 - sessions_organic
 - sessions_ai
 - missing_count
 - gsc_avg_position_missing
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing
 - ctr
 - sec_per_click
 - ai_share
 - engagement_per_organic_session

HIGH FEATURE-TO-FEATURE CORRELATION
('gsc_clicks', 'sessions_organic', np.float64(0.978))
('missing_count', 'ga4_total_engagement_sec_missing', np.float64(0.968))
('missing_count', 'sessions_organic_missing', np.float64(0.968))
('missing_count', 'sessions_ai_missing', np.float64(0.968))
('ga4_total_engagement_sec_missing', 'sessions_organic_missing', np.float64(1.0))
('ga4_total_engagement_sec_missing', 'sessions_ai_missing', np.float64(1.0))
('sessions_organic_missing', 'sessions_ai_missing', np.float64(1.0))


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



VIF TEST


,feature,VIF
0,missing_count,inf
1,ga4_total_engagement_sec_missing,inf
2,gsc_avg_position_missing,inf
3,sessions_ai_missing,inf
4,sessions_organic_missing,inf
5,sessions_organic,4.518367
6,gsc_clicks,3.863913
7,gsc_impressions,2.184852
8,ga4_total_engagement_sec,2.176906
9,sessions_ai,1.924955


**Block 3.5: keep selected impactful  features after VIF and correlation test**

In [16]:
# ============================================================
# ML-05 → ML-06/07 PREPARATION
# FINAL FEATURE SET + 90-DAY ELIGIBILITY CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL FEATURE DATASET PREPARATION")
print("=" * 70)

# ============================================================
# 1. FIND THE CURRENT 19-COLUMN DATAFRAME
# ============================================================

# Try known dataframe names first
candidate_names = [
    "df_baseline",
    "df_model_base",
    "df_features_clean",
    "df_clean",
    "df"
]

source_df = None
source_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            source_df = obj.copy()
            source_name = name
            break

if source_df is None:
    raise NameError(
        "No suitable dataframe found. "
        "Please load your 19-column Parquet dataset first."
    )

print(f"Source dataframe used: {source_name}")
print(f"Rows before cleaning: {len(source_df):,}")
print(f"Columns before cleaning: {source_df.shape[1]}")

# ============================================================
# 2. START FROM SOURCE DATA
# ============================================================

df_model_base = source_df.copy()

# Convert month safely
df_model_base["month"] = pd.to_datetime(
    df_model_base["month"],
    errors="coerce"
)

# ============================================================
# 3. REMOVE REDUNDANT MISSINGNESS FEATURES
# ============================================================

remove_columns = [
    "missing_count",
    "ga4_total_engagement_sec_missing",
    "sessions_organic_missing",
    "sessions_ai_missing"
]

# Only remove columns that actually exist
remove_columns = [
    col
    for col in remove_columns
    if col in df_model_base.columns
]

df_model_base = df_model_base.drop(
    columns=remove_columns
)

print("\n" + "=" * 70)
print("REMOVED REDUNDANT FEATURES")
print("=" * 70)

if remove_columns:
    for col in remove_columns:
        print(" -", col)
else:
    print("None")

# ============================================================
# 4. REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "content_hash_id",
    "month"
]

missing_required = [
    col
    for col in required_columns
    if col not in df_model_base.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing: {missing_required}"
    )

# ============================================================
# 5. REMOVE INVALID MONTH ROWS
# ============================================================

before_month_filter = len(df_model_base)

df_model_base = df_model_base[
    df_model_base["month"].notna()
].copy()

removed_invalid_months = (
    before_month_filter -
    len(df_model_base)
)

print(
    f"\nRows removed because month was invalid: "
    f"{removed_invalid_months:,}"
)

# ============================================================
# 6. REMOVE DUPLICATE PAGE-MONTH RECORDS IF ANY
# ============================================================

duplicate_count = (
    df_model_base
    .duplicated(
        subset=["content_hash_id", "month"]
    )
    .sum()
)

print(
    f"Duplicate page-month rows found: "
    f"{duplicate_count:,}"
)

if duplicate_count > 0:
    df_model_base = (
        df_model_base
        .drop_duplicates(
            subset=["content_hash_id", "month"],
            keep="first"
        )
        .copy()
    )

# ============================================================
# 7. SORT PAGE HISTORIES CHRONOLOGICALLY
# ============================================================

df_model_base = (
    df_model_base
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 8. CALCULATE OBSERVED HISTORY
# ============================================================

first_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("max")
)

df_model_base["content_age_days"] = (
    last_month - first_month
).dt.days

# ============================================================
# 9. 90-DAY ELIGIBILITY
# ============================================================

eligible_mask = (
    df_model_base["content_age_days"] >= 90
)

df_model_eligible = (
    df_model_base.loc[eligible_mask]
    .copy()
)

removed_rows = (
    len(df_model_base) -
    len(df_model_eligible)
)

print("\n" + "=" * 70)
print("90-DAY ELIGIBILITY")
print("=" * 70)

print(
    f"Rows before 90-day filter : "
    f"{len(df_model_base):,}"
)

print(
    f"Rows after 90-day filter  : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Rows removed              : "
    f"{removed_rows:,}"
)

# ============================================================
# 10. PAGE-LEVEL ELIGIBILITY CHECK
# ============================================================

page_age = (
    df_model_eligible
    .groupby("content_hash_id")["content_age_days"]
    .max()
)

if len(page_age) > 0:

    print(
        f"\nEligible pages: "
        f"{len(page_age):,}"
    )

    print(
        f"Minimum eligible history: "
        f"{page_age.min()} days"
    )

    print(
        f"Pages with <90 days remaining: "
        f"{(page_age < 90).sum():,}"
    )

    # Safety check
    assert (
        page_age >= 90
    ).all(), (
        "ERROR: A page below 90 days remains."
    )

else:
    raise ValueError(
        "No pages remain after the 90-day eligibility filter."
    )

# ============================================================
# 11. FINAL CHRONOLOGICAL ORDER
# ============================================================

df_model_eligible = (
    df_model_eligible
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 12. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Final rows    : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Final columns : "
    f"{df_model_eligible.shape[1]}"
)

print(
    f"Final shape   : "
    f"{df_model_eligible.shape}"
)

print("\nRemaining columns:")

for i, col in enumerate(
    df_model_eligible.columns,
    start=1
):
    print(f"{i:2}. {col}")

# ============================================================
# 13. FINAL 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET — FIRST 5 ROWS")
print("=" * 70)

display(
    df_model_eligible.head(5)
)

# ============================================================
# 14. FINAL 90-DAY SAFETY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL SAFETY CHECK")
print("=" * 70)

print(
    "Minimum content_age_days:",
    df_model_eligible["content_age_days"].min()
)

print(
    "Rows below 90 days:",
    (
        df_model_eligible["content_age_days"] < 90
    ).sum()
)

assert (
    df_model_eligible["content_age_days"] >= 90
).all()

print("PASS: No page below the 90-day requirement.")

# ============================================================
# 15. SAVE FINAL INTERMEDIATE DATASET
# ============================================================

output_path = (
    "/content/final_model_eligible.parquet"
)

df_model_eligible.to_parquet(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("PARQUET SAVED")
print("=" * 70)

print(output_path)

print("\n" + "=" * 70)
print("PREPARATION COMPLETE")
print("=" * 70)

FINAL FEATURE DATASET PREPARATION
Source dataframe used: df_model_base
Rows before cleaning: 2,871,202
Columns before cleaning: 15

REMOVED REDUNDANT FEATURES
None

Rows removed because month was invalid: 0
Duplicate page-month rows found: 0

90-DAY ELIGIBILITY
Rows before 90-day filter : 2,871,202
Rows after 90-day filter  : 2,705,303
Rows removed              : 165,899

Eligible pages: 349,557
Minimum eligible history: 92 days
Pages with <90 days remaining: 0

FINAL DATASET
Final rows    : 2,705,303
Final columns : 15
Final shape   : (2705303, 15)

Remaining columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. gsc_avg_position_missing
11. ctr
12. sec_per_click
13. ai_share
14. engagement_per_organic_session
15. content_age_days

FINAL DATASET — FIRST 5 ROWS


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,gsc_avg_position_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session,content_age_days
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0.006849,0.0,0.0,0.0,457
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457



FINAL SAFETY CHECK
Minimum content_age_days: 92
Rows below 90 days: 0
PASS: No page below the 90-day requirement.

PARQUET SAVED
/content/final_model_eligible.parquet

PREPARATION COMPLETE


**Block 4.5 — Target validation**

0 = DOWN → <= -30%
1 = FLAT → > -30% and < +50%
2 = UP → >= +50%

In [24]:
# ============================================================
# BLOCK 4.2 — FINAL FROZEN TARGET CREATION
#
# FINAL TARGET RULE
#   DOWN : future impression change <= -30%
#   FLAT : -30% < change < +50%
#   UP   : future impression change >= +50%
#
# ENCODING
#   0 = DOWN
#   1 = FLAT
#   2 = UP
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — FINAL FROZEN TARGET CREATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. SAFETY CHECK
# ------------------------------------------------------------

required_column = "future_impression_change_pct"

if required_column not in out.columns:
    raise KeyError(
        f"Required column '{required_column}' not found in out."
    )

# Work on a copy so original rolling dataset remains unchanged
df_target = out.copy()

# Ensure numeric
df_target[required_column] = pd.to_numeric(
    df_target[required_column],
    errors="coerce"
)

# Remove invalid future-change rows
before = len(df_target)

df_target = df_target[
    df_target[required_column].notna()
].copy()

after = len(df_target)

print(f"Rows before target validation : {before:,}")
print(f"Rows after invalid removal    : {after:,}")
print(f"Rows removed                  : {before - after:,}")

# ------------------------------------------------------------
# 2. FINAL FROZEN THRESHOLDS
# ------------------------------------------------------------

DOWN_THRESHOLD = -30
UP_THRESHOLD = 50

print("\n" + "=" * 80)
print("FINAL FROZEN TARGET RULE")
print("=" * 80)

print("DOWN : future impression change <= -30%")
print("FLAT : -30% < future impression change < +50%")
print("UP   : future impression change >= +50%")

# ------------------------------------------------------------
# 3. CREATE NUMERIC TARGET
#
# IMPORTANT:
#   0 = DOWN
#   1 = FLAT
#   2 = UP
# ------------------------------------------------------------

df_target["target"] = np.select(
    [
        df_target[required_column] <= DOWN_THRESHOLD,
        df_target[required_column] >= UP_THRESHOLD
    ],
    [
        0,
        2
    ],
    default=1
).astype("int8")

# ------------------------------------------------------------
# 4. CREATE HUMAN-READABLE TARGET LABEL
# ------------------------------------------------------------

label_mapping = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

df_target["target_label"] = (
    df_target["target"]
    .map(label_mapping)
)

# ------------------------------------------------------------
# 5. TARGET ENCODING SAFETY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET ENCODING")
print("=" * 80)

print("0 = DOWN")
print("1 = FLAT")
print("2 = UP")

assert set(
    df_target["target"].unique()
).issubset({0, 1, 2})

assert (
    df_target["target_label"].notna().all()
)

assert (
    df_target["target_label"].isin(
        ["DOWN", "FLAT", "UP"]
    ).all()
)

print("\n✓ Encoding confirmed.")
print("✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")

# ------------------------------------------------------------
# 6. FINAL TARGET DISTRIBUTION
# ------------------------------------------------------------

target_summary = (
    df_target
    .groupby(
        ["target", "target_label"],
        sort=True
    )
    .size()
    .reset_index(
        name="count"
    )
)

target_summary["percentage"] = (
    target_summary["count"]
    / len(df_target)
    * 100
).round(2)

print("\n" + "=" * 80)
print("FINAL TARGET DISTRIBUTION")
print("=" * 80)

display(target_summary)

# ------------------------------------------------------------
# 7. TARGET COUNTS
# ------------------------------------------------------------

down_count = int(
    (df_target["target"] == 0).sum()
)

flat_count = int(
    (df_target["target"] == 1).sum()
)

up_count = int(
    (df_target["target"] == 2).sum()
)

print("\n" + "=" * 80)
print("FINAL TARGET COUNTS")
print("=" * 80)

print(
    f"DOWN : {down_count:,} "
    f"({down_count / len(df_target) * 100:.2f}%)"
)

print(
    f"FLAT : {flat_count:,} "
    f"({flat_count / len(df_target) * 100:.2f}%)"
)

print(
    f"UP   : {up_count:,} "
    f"({up_count / len(df_target) * 100:.2f}%)"
)

print(
    f"\nTotal: {len(df_target):,}"
)

# ------------------------------------------------------------
# 8. TARGET BOUNDARY SANITY CHECK
# ------------------------------------------------------------

change = df_target[
    "future_impression_change_pct"
]

# Every DOWN must be <= -30
assert (
    df_target.loc[
        df_target["target"] == 0,
        required_column
    ] <= -30
).all()

# Every UP must be >= +50
assert (
    df_target.loc[
        df_target["target"] == 2,
        required_column
    ] >= 50
).all()

# Every FLAT must be between -30 and +50
flat_values = df_target.loc[
    df_target["target"] == 1,
    required_column
]

assert (
    (flat_values > -30)
    & (flat_values < 50)
).all()

print("\n" + "=" * 80)
print("BOUNDARY SANITY CHECK")
print("=" * 80)

print("✓ DOWN contains only change <= -30%")
print("✓ FLAT contains only -30% < change < +50%")
print("✓ UP contains only change >= +50%")
print("✓ Every row has exactly one target class")

# ------------------------------------------------------------
# 9. PREVIEW
# ------------------------------------------------------------

preview_columns = [
    "content_hash_id",
    "window_start",
    "window_end",
    "future_start",
    "future_end",
    "current_imp_3m",
    "future_imp_3m",
    "future_impression_change_pct",
    "target",
    "target_label"
]

preview_columns = [
    c
    for c in preview_columns
    if c in df_target.columns
]

print("\n" + "=" * 80)
print("TARGET DATASET PREVIEW")
print("=" * 80)

display(
    df_target[
        preview_columns
    ].head(10)
)

# ------------------------------------------------------------
# 10. FINAL OBJECT
# ------------------------------------------------------------

# This is now the authoritative rolling-window dataset
# for the next step.
out_final_target = df_target.copy()

print("\n" + "=" * 80)
print("BLOCK 4.2 COMPLETE")
print("=" * 80)

print(
    f"Final rows : {len(out_final_target):,}"
)

print(
    "Frozen target:"
)

print(
    "0 = DOWN  (<= -30%)"
)

print(
    "1 = FLAT  (-30% to < +50%)"
)

print(
    "2 = UP    (>= +50%)"
)

print(
    "\n✓ Target definition is now frozen."
)

W04 — FINAL FROZEN TARGET CREATION
Rows before target validation : 626,836
Rows after invalid removal    : 626,836
Rows removed                  : 0

FINAL FROZEN TARGET RULE
DOWN : future impression change <= -30%
FLAT : -30% < future impression change < +50%
UP   : future impression change >= +50%

TARGET ENCODING
0 = DOWN
1 = FLAT
2 = UP

✓ Encoding confirmed.
✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP

FINAL TARGET DISTRIBUTION


,target,target_label,count,percentage
0,0,DOWN,206390,32.93
1,1,FLAT,156653,24.99
2,2,UP,263793,42.08



FINAL TARGET COUNTS
DOWN : 206,390 (32.93%)
FLAT : 156,653 (24.99%)
UP   : 263,793 (42.08%)

Total: 626,836

BOUNDARY SANITY CHECK
✓ DOWN contains only change <= -30%
✓ FLAT contains only -30% < change < +50%
✓ UP contains only change >= +50%
✓ Every row has exactly one target class

TARGET DATASET PREVIEW


,content_hash_id,window_start,window_end,future_start,future_end,current_imp_3m,future_imp_3m,future_impression_change_pct,target,target_label
0,content_000005d4ced12088,2025-03-01,2025-05-01,2025-06-01,2025-08-01,136.666667,378.333333,176.829268,2,UP
1,content_000005d4ced12088,2025-04-01,2025-06-01,2025-07-01,2025-09-01,180.666667,506.666667,180.442804,2,UP
2,content_000005d4ced12088,2025-05-01,2025-07-01,2025-08-01,2025-10-01,216.666667,487.666667,125.076923,2,UP
3,content_000005d4ced12088,2025-06-01,2025-08-01,2025-09-01,2025-11-01,378.333333,280.333333,-25.903084,1,FLAT
4,content_000005d4ced12088,2025-07-01,2025-09-01,2025-10-01,2025-12-01,506.666667,167.000000,-67.039474,0,DOWN
5,content_000005d4ced12088,2025-08-01,2025-10-01,2025-11-01,2026-01-01,487.666667,106.333333,-78.195489,0,DOWN
6,content_000005d4ced12088,2025-09-01,2025-11-01,2025-12-01,2026-02-01,280.333333,74.333333,-73.483948,0,DOWN
7,content_000005d4ced12088,2025-10-01,2025-12-01,2026-01-01,2026-03-01,167.000000,41.666667,-75.049900,0,DOWN
8,content_000005d4ced12088,2025-11-01,2026-01-01,2026-02-01,2026-04-01,106.333333,63.666667,-40.125392,0,DOWN
9,content_000005d4ced12088,2025-12-01,2026-02-01,2026-03-01,2026-05-01,74.333333,82.666667,11.210762,1,FLAT



BLOCK 4.2 COMPLETE
Final rows : 626,836
Frozen target:
0 = DOWN  (<= -30%)
1 = FLAT  (-30% to < +50%)
2 = UP    (>= +50%)

✓ Target definition is now frozen.


**Block 4 — Rolling 90-day features + target**

In [27]:
# ============================================================
# BLOCK 4A — ROLLING 3M FEATURES + FUTURE 3M TARGET DATA
#
# CURRENT 3 MONTHS = MODEL INPUT FEATURES
# NEXT 3 MONTHS    = TARGET GENERATION ONLY
#
# IMPORTANT:
# - Future columns are retained in this intermediate dataset
# - target_label is NOT created here
# - Target threshold is FROZEN:
#       <= -30%  -> DOWN
#       > -30% and < +50% -> FLAT
#       >= +50% -> UP
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4A — ROLLING 3M FEATURES + FUTURE 3M TARGET GENERATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. FIND SOURCE DATAFRAME
# ------------------------------------------------------------

candidate_names = [
    "df_model_eligible",
    "df_model_base",
    "df_baseline",
    "df"
]

source_df = None
source_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            source_df = obj.copy()
            source_name = name
            break

if source_df is None:
    raise NameError(
        "No suitable source dataframe found. "
        "Expected df_model_eligible or another known dataframe."
    )

print(f"Source dataframe : {source_name}")
print(f"Source rows      : {len(source_df):,}")
print(f"Source columns   : {source_df.shape[1]}")

df_roll = source_df.copy()

# ------------------------------------------------------------
# 2. DATE CLEANING
# ------------------------------------------------------------

df_roll["month"] = pd.to_datetime(
    df_roll["month"],
    errors="coerce"
)

df_roll = df_roll.dropna(
    subset=["content_hash_id", "month"]
).copy()

# ------------------------------------------------------------
# 3. REMOVE DUPLICATE PAGE-MONTH RECORDS
# ------------------------------------------------------------

before_dup = len(df_roll)

df_roll = (
    df_roll
    .drop_duplicates(
        subset=["content_hash_id", "month"],
        keep="first"
    )
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

print(
    f"Duplicate rows removed: "
    f"{before_dup - len(df_roll):,}"
)

# ------------------------------------------------------------
# 4. REQUIRED RAW FEATURES
#
# ai_other_missing IS NOT REQUIRED because it does not exist
# in the current 15-column source dataset.
# ------------------------------------------------------------

raw_feature_columns = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "gsc_avg_position_missing",
    "ctr",
    "sec_per_click",
    "ai_share",
    "engagement_per_organic_session"
]

missing_features = [
    col
    for col in raw_feature_columns
    if col not in df_roll.columns
]

if missing_features:
    raise KeyError(
        "Required raw feature columns are missing:\n"
        + "\n".join(missing_features)
    )

print("\nAll required raw feature columns found.")

# ------------------------------------------------------------
# 5. MONTH PERIOD
# ------------------------------------------------------------

df_roll["month_period"] = (
    df_roll["month"].dt.to_period("M")
)

# ------------------------------------------------------------
# 6. VALID 6-MONTH CONTINUOUS WINDOWS
#
# Current:
# M0 M1 M2
#
# Future:
# M3 M4 M5
# ------------------------------------------------------------

page = df_roll["content_hash_id"]
period = df_roll["month_period"]

valid_6m = (
    page.eq(page.shift(-1))
    & page.eq(page.shift(-2))
    & page.eq(page.shift(-3))
    & page.eq(page.shift(-4))
    & page.eq(page.shift(-5))

    & period.add(1).eq(period.shift(-1))
    & period.add(2).eq(period.shift(-2))
    & period.add(3).eq(period.shift(-3))
    & period.add(4).eq(period.shift(-4))
    & period.add(5).eq(period.shift(-5))
)

valid_idx = np.flatnonzero(
    valid_6m.to_numpy()
)

print("\n" + "=" * 80)
print("VALID 6-MONTH WINDOWS")
print("=" * 80)

print(
    f"Valid current + future windows: "
    f"{len(valid_idx):,}"
)

# ------------------------------------------------------------
# 7. CREATE OUTPUT
# ------------------------------------------------------------

out = pd.DataFrame(
    index=np.arange(len(valid_idx))
)

# ------------------------------------------------------------
# 8. METADATA
# ------------------------------------------------------------

out["content_hash_id"] = (
    df_roll["content_hash_id"]
    .iloc[valid_idx]
    .to_numpy()
)

if "client_hash_id" in df_roll.columns:

    out["client_hash_id"] = (
        df_roll["client_hash_id"]
        .iloc[valid_idx]
        .to_numpy()
    )

out["window_start"] = (
    df_roll["month"]
    .iloc[valid_idx]
    .to_numpy()
)

out["window_end"] = (
    df_roll["month"]
    .iloc[valid_idx + 2]
    .to_numpy()
)

# ------------------------------------------------------------
# 9. FUTURE WINDOW DATES
# ------------------------------------------------------------

out["future_start"] = (
    df_roll["month"]
    .iloc[valid_idx + 3]
    .to_numpy()
)

out["future_end"] = (
    df_roll["month"]
    .iloc[valid_idx + 5]
    .to_numpy()
)

# ------------------------------------------------------------
# 10. CURRENT 3-MONTH FEATURES
#
# For every raw feature:
# mean_3m = M0 + M1 + M2 / 3
# last    = M2
# ------------------------------------------------------------

for col in raw_feature_columns:

    values = pd.to_numeric(
        df_roll[col],
        errors="coerce"
    ).to_numpy()

    v0 = values[valid_idx]
    v1 = values[valid_idx + 1]
    v2 = values[valid_idx + 2]

    out[f"{col}_mean_3m"] = (
        v0 + v1 + v2
    ) / 3

    out[f"{col}_last"] = v2

# ------------------------------------------------------------
# 11. CURRENT + FUTURE IMPRESSIONS
# ------------------------------------------------------------

imp = pd.to_numeric(
    df_roll["gsc_impressions"],
    errors="coerce"
).to_numpy()

current_imp_3m = (
    imp[valid_idx]
    + imp[valid_idx + 1]
    + imp[valid_idx + 2]
) / 3

future_imp_3m = (
    imp[valid_idx + 3]
    + imp[valid_idx + 4]
    + imp[valid_idx + 5]
) / 3

out["current_imp_3m"] = current_imp_3m

# Future value is intentionally retained
# because this block is target-generation stage.
out["future_imp_3m"] = future_imp_3m

# ------------------------------------------------------------
# 12. FUTURE IMPRESSION CHANGE
# ------------------------------------------------------------

valid_change = (
    np.isfinite(current_imp_3m)
    & np.isfinite(future_imp_3m)
    & (current_imp_3m > 0)
)

out["future_impression_change_pct"] = np.nan

out.loc[
    valid_change,
    "future_impression_change_pct"
] = (
    (
        future_imp_3m[valid_change]
        - current_imp_3m[valid_change]
    )
    / current_imp_3m[valid_change]
) * 100

# Remove rows where target cannot be calculated
out = out[
    out["future_impression_change_pct"].notna()
].copy()

# ------------------------------------------------------------
# 13. FROZEN TARGET RULE
#
# VALIDATED RULE:
#
# <= -30%       = DOWN
# -30% to <50%  = FLAT
# >= +50%       = UP
# ------------------------------------------------------------

out["target"] = np.select(
    [
        out["future_impression_change_pct"] <= -30,
        out["future_impression_change_pct"] >= 50
    ],
    [
        "Down",
        "Up"
    ],
    default="Flat"
)

# ------------------------------------------------------------
# 14. TARGET DISTRIBUTION
# ------------------------------------------------------------

target_summary = (
    out["target"]
    .value_counts()
    .reindex(
        ["Down", "Flat", "Up"],
        fill_value=0
    )
    .rename_axis("target")
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"]
    / len(out)
    * 100
).round(2)

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

display(target_summary)

# ------------------------------------------------------------
# 15. SAFETY CHECK
# ------------------------------------------------------------

change = out[
    "future_impression_change_pct"
]

assert (
    out.loc[
        change <= -30,
        "target"
    ] == "Down"
).all()

assert (
    out.loc[
        (change > -30) & (change < 50),
        "target"
    ] == "Flat"
).all()

assert (
    out.loc[
        change >= 50,
        "target"
    ] == "Up"
).all()

print("\n✓ DOWN threshold confirmed: <= -30%")
print("✓ FLAT threshold confirmed: > -30% and < +50%")
print("✓ UP threshold confirmed: >= +50%")

# ------------------------------------------------------------
# 16. CURRENT FEATURE COUNT
# ------------------------------------------------------------

current_feature_columns = [
    col
    for col in out.columns
    if (
        col.endswith("_mean_3m")
        or col.endswith("_last")
    )
]

# current_imp_3m is additional current feature
if "current_imp_3m" in out.columns:
    current_feature_columns.append(
        "current_imp_3m"
    )

print("\n" + "=" * 80)
print("CURRENT MODEL FEATURES")
print("=" * 80)

print(
    f"Current feature columns: "
    f"{len(current_feature_columns)}"
)

for i, col in enumerate(
    current_feature_columns,
    1
):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# 17. INTERMEDIATE DATASET
#
# Includes:
# metadata
# current features
# future target-generation data
# target
#
# target_label comes in next block.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERMEDIATE ROLLING DATASET")
print("=" * 80)

print(
    f"Rows    : {len(out):,}"
)

print(
    f"Columns : {out.shape[1]}"
)

display(out.head(10))

# ------------------------------------------------------------
# 18. SAVE INTERMEDIATE DATASET
# ------------------------------------------------------------

rolling_target_path = (
    "/content/rolling_3m_future_3m_target_dataset.parquet"
)

out.to_parquet(
    rolling_target_path,
    index=False
)

print("\nSaved:")
print(rolling_target_path)

print("\n" + "=" * 80)
print("BLOCK 4A COMPLETE")
print("=" * 80)

BLOCK 4A — ROLLING 3M FEATURES + FUTURE 3M TARGET GENERATION
Source dataframe : df_model_eligible
Source rows      : 2,705,303
Source columns   : 15
Duplicate rows removed: 0

All required raw feature columns found.

VALID 6-MONTH WINDOWS
Valid current + future windows: 978,801

TARGET DISTRIBUTION


,target,count,percentage
0,Down,206390,32.93
1,Flat,156653,24.99
2,Up,263793,42.08



✓ DOWN threshold confirmed: <= -30%
✓ FLAT threshold confirmed: > -30% and < +50%
✓ UP threshold confirmed: >= +50%

CURRENT MODEL FEATURES
Current feature columns: 23
 1. gsc_clicks_mean_3m
 2. gsc_clicks_last
 3. gsc_impressions_mean_3m
 4. gsc_impressions_last
 5. gsc_avg_position_mean_3m
 6. gsc_avg_position_last
 7. ga4_total_engagement_sec_mean_3m
 8. ga4_total_engagement_sec_last
 9. sessions_organic_mean_3m
10. sessions_organic_last
11. sessions_ai_mean_3m
12. sessions_ai_last
13. gsc_avg_position_missing_mean_3m
14. gsc_avg_position_missing_last
15. ctr_mean_3m
16. ctr_last
17. sec_per_click_mean_3m
18. sec_per_click_last
19. ai_share_mean_3m
20. ai_share_last
21. engagement_per_organic_session_mean_3m
22. engagement_per_organic_session_last
23. current_imp_3m

INTERMEDIATE ROLLING DATASET
Rows    : 626,836
Columns : 32


,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,sec_per_click_mean_3m,sec_per_click_last,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,future_imp_3m,future_impression_change_pct,target
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,0.0,0.0,0.0,0.0,136.666667,378.333333,176.829268,Up
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,0.0,0.0,0.0,0.0,180.666667,506.666667,180.442804,Up
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,0.0,0.0,0.0,0.0,216.666667,487.666667,125.076923,Up
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,0.0,0.0,0.0,0.0,378.333333,280.333333,-25.903084,Flat
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,0.0,0.0,0.0,0.0,506.666667,167.000000,-67.039474,Down
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.666667,0.0,487.666667,197.0,...,0.0,0.0,0.0,0.0,0.0,0.0,487.666667,106.333333,-78.195489,Down
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.333333,0.0,280.333333,120.0,...,0.0,0.0,0.0,0.0,0.0,0.0,280.333333,74.333333,-73.483948,Down
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,167.000000,184.0,...,0.0,0.0,0.0,0.0,0.0,0.0,167.000000,41.666667,-75.049900,Down
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.000000,0.0,106.333333,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,106.333333,63.666667,-40.125392,Down
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,2026-02-01,2026-03-01,2026-05-01,0.000000,0.0,74.333333,24.0,...,0.0,0.0,0.0,0.0,0.0,0.0,74.333333,82.666667,11.210762,Flat



Saved:
/content/rolling_3m_future_3m_target_dataset.parquet

BLOCK 4A COMPLETE


In [28]:
out.tail(10)

,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,sec_per_click_mean_3m,sec_per_click_last,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,future_imp_3m,future_impression_change_pct,target
978791,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.666667,0.0,375.666667,304.0,...,0.0,0.0,0.0,0.0,0.0,0.0,375.666667,59.000000,-84.294587,Down
978792,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.666667,0.0,333.666667,69.0,...,0.0,0.0,0.0,0.0,0.0,0.0,333.666667,42.000000,-87.412587,Down
978793,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.000000,0.0,154.666667,91.0,...,0.0,0.0,0.0,0.0,0.0,0.0,154.666667,19.666667,-87.284483,Down
978794,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.000000,0.0,59.000000,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,59.000000,26.666667,-54.802260,Down
978795,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.000000,0.0,42.000000,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,42.000000,1543.666667,3575.396825,Up
978796,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.000000,0.0,19.666667,24.0,...,0.0,0.0,0.0,0.0,0.0,0.0,19.666667,2473.333333,12476.271186,Up
978797,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,26.666667,38.0,...,0.0,0.0,0.0,0.0,0.0,0.0,26.666667,2648.333333,9831.250000,Up
978798,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.333333,1.0,1543.666667,4569.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1543.666667,1747.000000,13.172101,Flat
978799,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-12-01,2026-02-01,2026-03-01,2026-05-01,1.000000,2.0,2473.333333,2813.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2473.333333,937.666667,-62.088949,Down
978800,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-01-01,2026-03-01,2026-04-01,2026-06-01,1.000000,0.0,2648.333333,563.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2648.333333,799.666667,-69.804909,Down


**Categorical encoding**

In [29]:
# ============================================================
# BLOCK 4B — TARGET CATEGORICAL ENCODING
#
# Frozen mapping:
#   0 = DOWN
#   1 = FLAT
#   2 = UP
#
# Creates:
#   target       -> integer
#   target_label -> readable label
# ============================================================

print("=" * 80)
print("BLOCK 4B — TARGET CATEGORICAL ENCODING")
print("=" * 80)

df_target_encoded = out.copy()

# ------------------------------------------------------------
# 1. ORIGINAL TARGET CHECK
# ------------------------------------------------------------

allowed_targets = {
    "Down",
    "Flat",
    "Up"
}

actual_targets = set(
    df_target_encoded["target"]
    .dropna()
    .unique()
)

print("Original target values:")
print(sorted(actual_targets))

assert actual_targets.issubset(
    allowed_targets
), (
    f"Unexpected target values: "
    f"{actual_targets - allowed_targets}"
)

# ------------------------------------------------------------
# 2. KEEP READABLE LABEL
# ------------------------------------------------------------

df_target_encoded["target_label"] = (
    df_target_encoded["target"]
    .copy()
)

# ------------------------------------------------------------
# 3. ENCODE TARGET
# ------------------------------------------------------------

target_mapping = {
    "Down": 0,
    "Flat": 1,
    "Up": 2
}

df_target_encoded["target"] = (
    df_target_encoded["target"]
    .map(target_mapping)
    .astype("int8")
)

# ------------------------------------------------------------
# 4. VERIFY ENCODING
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET ENCODING")
print("=" * 80)

print("0 = DOWN")
print("1 = FLAT")
print("2 = UP")

assert set(
    df_target_encoded["target"].unique()
) == {0, 1, 2}

assert (
    df_target_encoded.loc[
        df_target_encoded["target"] == 0,
        "target_label"
    ] == "Down"
).all()

assert (
    df_target_encoded.loc[
        df_target_encoded["target"] == 1,
        "target_label"
    ] == "Flat"
).all()

assert (
    df_target_encoded.loc[
        df_target_encoded["target"] == 2,
        "target_label"
    ] == "Up"
).all()

print("\n✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")
print("✓ Encoding verified")

# ------------------------------------------------------------
# 5. DISTRIBUTION
# ------------------------------------------------------------

encoding_summary = (
    df_target_encoded
    .groupby(
        ["target", "target_label"]
    )
    .size()
    .reset_index(
        name="count"
    )
)

encoding_summary["percentage"] = (
    encoding_summary["count"]
    / len(df_target_encoded)
    * 100
).round(2)

display(
    encoding_summary
)

# ------------------------------------------------------------
# 6. PREVIEW
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ENCODED DATASET PREVIEW")
print("=" * 80)

display(
    df_target_encoded.head(10)
)

# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

encoded_path = (
    "/content/rolling_3m_target_encoded.parquet"
)

df_target_encoded.to_parquet(
    encoded_path,
    index=False
)

print("\nSaved:")
print(encoded_path)

print("\n" + "=" * 80)
print("BLOCK 4B COMPLETE")
print("=" * 80)

BLOCK 4B — TARGET CATEGORICAL ENCODING
Original target values:
['Down', 'Flat', 'Up']

TARGET ENCODING
0 = DOWN
1 = FLAT
2 = UP

✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP
✓ Encoding verified


,target,target_label,count,percentage
0,0,Down,206390,32.93
1,1,Flat,156653,24.99
2,2,Up,263793,42.08



ENCODED DATASET PREVIEW


,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,sec_per_click_last,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,future_imp_3m,future_impression_change_pct,target,target_label
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,0.0,0.0,0.0,136.666667,378.333333,176.829268,2,Up
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,0.0,0.0,0.0,180.666667,506.666667,180.442804,2,Up
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,0.0,0.0,0.0,216.666667,487.666667,125.076923,2,Up
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,0.0,0.0,0.0,378.333333,280.333333,-25.903084,1,Flat
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,0.0,0.0,0.0,506.666667,167.000000,-67.039474,0,Down
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.666667,0.0,487.666667,197.0,...,0.0,0.0,0.0,0.0,0.0,487.666667,106.333333,-78.195489,0,Down
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.333333,0.0,280.333333,120.0,...,0.0,0.0,0.0,0.0,0.0,280.333333,74.333333,-73.483948,0,Down
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,167.000000,184.0,...,0.0,0.0,0.0,0.0,0.0,167.000000,41.666667,-75.049900,0,Down
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.000000,0.0,106.333333,15.0,...,0.0,0.0,0.0,0.0,0.0,106.333333,63.666667,-40.125392,0,Down
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,2026-02-01,2026-03-01,2026-05-01,0.000000,0.0,74.333333,24.0,...,0.0,0.0,0.0,0.0,0.0,74.333333,82.666667,11.210762,1,Flat



Saved:
/content/rolling_3m_target_encoded.parquet

BLOCK 4B COMPLETE


In [30]:
df_target_encoded.tail(5)

,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,sec_per_click_last,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,future_imp_3m,future_impression_change_pct,target,target_label
978796,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.000000,0.0,19.666667,24.0,...,0.0,0.0,0.0,0.0,0.0,19.666667,2473.333333,12476.271186,2,Up
978797,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,26.666667,38.0,...,0.0,0.0,0.0,0.0,0.0,26.666667,2648.333333,9831.250000,2,Up
978798,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.333333,1.0,1543.666667,4569.0,...,0.0,0.0,0.0,0.0,0.0,1543.666667,1747.000000,13.172101,1,Flat
978799,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2025-12-01,2026-02-01,2026-03-01,2026-05-01,1.000000,2.0,2473.333333,2813.0,...,0.0,0.0,0.0,0.0,0.0,2473.333333,937.666667,-62.088949,0,Down
978800,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-01-01,2026-03-01,2026-04-01,2026-06-01,1.000000,0.0,2648.333333,563.0,...,0.0,0.0,0.0,0.0,0.0,2648.333333,799.666667,-69.804909,0,Down


**Freeze Final Threshold**

In [10]:
# ============================================================
# W04 — BLOCK 4.7
# FINAL TARGET THRESHOLD SELECTION
#
# FINAL FROZEN RULE:
#
# DOWN : <= -30%
# FLAT : > -30% and < +50%
# UP   : >= +50%
#
# Reason:
# -20% / +20% produced too many UP labels.
# The observed growth distribution is highly right-skewed.
# Therefore asymmetric thresholds are more appropriate.
#
# This block ONLY evaluates and freezes the target rule.
# Final target encoding will be done in the next block.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — FINAL TARGET THRESHOLD SELECTION")
print("=" * 80)


# ============================================================
# 1. WORK ON COPY
# ============================================================

df_target_check = df_windows.copy()

change = pd.to_numeric(
    df_target_check["future_impression_change_pct"],
    errors="coerce"
)

change = change.replace(
    [np.inf, -np.inf],
    np.nan
)


valid_change = change.dropna()

print(
    f"\nValid future windows: "
    f"{len(valid_change):,}"
)


# ============================================================
# 2. FINAL FROZEN THRESHOLDS
# ============================================================

DOWN_THRESHOLD = -30
UP_THRESHOLD = 50


# ============================================================
# 3. APPLY FINAL LABEL RULE
# ============================================================

final_labels = np.select(
    [
        valid_change <= DOWN_THRESHOLD,

        valid_change >= UP_THRESHOLD
    ],
    [
        "DOWN",
        "UP"
    ],
    default="FLAT"
)


final_label_series = pd.Series(
    final_labels,
    index=valid_change.index,
    name="future_direction"
)


# ============================================================
# 4. FINAL CLASS DISTRIBUTION
# ============================================================

label_summary = (
    final_label_series
    .value_counts()
    .reindex(
        ["DOWN", "FLAT", "UP"],
        fill_value=0
    )
    .rename_axis("target_label")
    .reset_index(name="count")
)

label_summary["percentage"] = (
    label_summary["count"]
    / len(final_label_series)
    * 100
).round(2)


print("\n" + "=" * 80)
print("FINAL TARGET DISTRIBUTION")
print("=" * 80)

display(label_summary)


# ============================================================
# 5. FINAL THRESHOLD EXPLANATION
# ============================================================

print("\n" + "=" * 80)
print("FINAL TARGET RULE")
print("=" * 80)

print(
    "DOWN : future impression change <= -30%"
)

print(
    "FLAT : -30% < future impression change < +50%"
)

print(
    "UP   : future impression change >= +50%"
)


# ============================================================
# 6. WHY THIS RULE WAS SELECTED
# ============================================================

print("\n" + "=" * 80)
print("WHY THIS RULE WAS SELECTED")
print("=" * 80)

print("""
1. A ±20% rule was rejected because +20% produced
   too many UP observations.

2. The future-impression distribution is strongly
   asymmetric, with very large positive changes.

3. A -30% DOWN threshold focuses the target on
   meaningful future decline.

4. A +50% UP threshold prevents ordinary/moderate
   growth from being automatically labelled UP.

5. The remaining observations are treated as FLAT,
   representing normal or non-actionable movement.

6. The rule is simple, transparent and reproducible.

7. The final ML target will be based ONLY on this
   frozen future outcome definition.
""")


# ============================================================
# 7. SANITY CHECK
# ============================================================

down_count = (
    final_label_series == "DOWN"
).sum()

flat_count = (
    final_label_series == "FLAT"
).sum()

up_count = (
    final_label_series == "UP"
).sum()

total = len(final_label_series)


print("\n" + "=" * 80)
print("SANITY CHECK")
print("=" * 80)

print(f"DOWN : {down_count:,} ({down_count / total * 100:.2f}%)")
print(f"FLAT : {flat_count:,} ({flat_count / total * 100:.2f}%)")
print(f"UP   : {up_count:,} ({up_count / total * 100:.2f}%)")

print(
    f"\nTotal : "
    f"{down_count + flat_count + up_count:,}"
)

if (
    down_count
    + flat_count
    + up_count
    == total
):
    print(
        "\n✓ Every valid future window received exactly one label."
    )
else:
    print(
        "\n⚠ Label count mismatch — review required."
    )


# ============================================================
# 8. FINAL VERDICT
# ============================================================

print("\n" + "=" * 80)
print("FINAL DECISION")
print("=" * 80)

print(
    "✓ FINAL THRESHOLD FROZEN"
)

print(
    "✓ DOWN = <= -30%"
)

print(
    "✓ FLAT = -30% to +50%"
)

print(
    "✓ UP = >= +50%"
)

print(
    "\nNext step: create the final target column "
    "and encode DOWN / FLAT / UP for ML training."
)

W04 — FINAL TARGET THRESHOLD SELECTION

Valid future windows: 357,342

FINAL TARGET DISTRIBUTION


,target_label,count,percentage
0,DOWN,113450,31.75
1,FLAT,66600,18.64
2,UP,177292,49.61



FINAL TARGET RULE
DOWN : future impression change <= -30%
FLAT : -30% < future impression change < +50%
UP   : future impression change >= +50%

WHY THIS RULE WAS SELECTED

1. A ±20% rule was rejected because +20% produced
   too many UP observations.

2. The future-impression distribution is strongly
   asymmetric, with very large positive changes.

3. A -30% DOWN threshold focuses the target on
   meaningful future decline.

4. A +50% UP threshold prevents ordinary/moderate
   growth from being automatically labelled UP.

5. The remaining observations are treated as FLAT,
   representing normal or non-actionable movement.

6. The rule is simple, transparent and reproducible.

7. The final ML target will be based ONLY on this
   frozen future outcome definition.


SANITY CHECK
DOWN : 113,450 (31.75%)
FLAT : 66,600 (18.64%)
UP   : 177,292 (49.61%)

Total : 357,342

✓ Every valid future window received exactly one label.

FINAL DECISION
✓ FINAL THRESHOLD FROZEN
✓ DOWN = <= -30%
✓ FLAT 

**Block 4.7 — Categorical Encoding of Targeting**

**Down**  → 0

**Flat**  → 1

**Up**   → 2

In [13]:
# ============================================================
# BLOCK 4.9 — FINAL MODEL DATASET
#
# FINAL DATASET = 30 COLUMNS
#
# 5 metadata columns
# + 23 model features
# + 2 target columns
#
# Metadata:
#   content_hash_id
#   client_hash_id
#   window_start
#   window_end
#   month
#
# Model features:
#   23 rolling/current features
#
# Target:
#   target       -> 0 / 1 / 2
#   target_label -> Down / Flat / Up
#
# NO future_* columns are retained.
# ============================================================

import pandas as pd

print("=" * 80)
print("W04 — FINAL MODEL DATASET")
print("=" * 80)

# ------------------------------------------------------------
# 1. EXACT 23 MODEL FEATURES
# ------------------------------------------------------------

model_features = [
    "gsc_clicks_mean_3m",
    "gsc_clicks_last",

    "gsc_impressions_mean_3m",
    "gsc_impressions_last",

    "gsc_avg_position_mean_3m",
    "gsc_avg_position_last",

    "ga4_total_engagement_sec_mean_3m",
    "ga4_total_engagement_sec_last",

    "sessions_organic_mean_3m",
    "sessions_organic_last",

    "sessions_ai_mean_3m",
    "sessions_ai_last",

    "gsc_avg_position_missing_mean_3m",
    "gsc_avg_position_missing_last",

    "ctr_mean_3m",
    "ctr_last",

    "sec_per_click_mean_3m",
    "sec_per_click_last",

    "ai_share_mean_3m",
    "ai_share_last",

    "engagement_per_organic_session_mean_3m",
    "engagement_per_organic_session_last",

    "current_imp_3m"
]

assert len(model_features) == 23

# ------------------------------------------------------------
# 2. METADATA COLUMNS
# ------------------------------------------------------------

metadata_columns = [
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end"
]

# ------------------------------------------------------------
# 3. CREATE MONTH COLUMN
#
# month = latest month of the current 3-month window
#
# Example:
# Jan + Feb + Mar
#       ↓
# month = Mar
# ------------------------------------------------------------

df_final_model = df_final_ml.copy()

df_final_model["month"] = pd.to_datetime(
    df_final_model["window_end"],
    errors="coerce"
)

# ------------------------------------------------------------
# 4. TARGET COLUMNS
# ------------------------------------------------------------

target_columns = [
    "target",
    "target_label"
]

# ------------------------------------------------------------
# 5. FINAL COLUMN ORDER
# ------------------------------------------------------------

final_columns = (
    metadata_columns
    + ["month"]
    + model_features
    + target_columns
)

# ------------------------------------------------------------
# 6. VERIFY ALL REQUIRED COLUMNS EXIST
# ------------------------------------------------------------

missing_required = [
    col
    for col in final_columns
    if col not in df_final_model.columns
]

if missing_required:

    raise KeyError(
        "Missing required columns:\n"
        + "\n".join(missing_required)
    )

# ------------------------------------------------------------
# 7. CREATE FINAL DATASET
# ------------------------------------------------------------

final_model_dataset = df_final_model[
    final_columns
].copy()

# ------------------------------------------------------------
# 8. FUTURE COLUMN CHECK
# ------------------------------------------------------------

future_columns = [
    col
    for col in final_model_dataset.columns
    if col.startswith("future_")
]

assert len(future_columns) == 0, (
    f"Future columns found: {future_columns}"
)

# ------------------------------------------------------------
# 9. FINAL SHAPE CHECK
# ------------------------------------------------------------

expected_columns = (
    5      # metadata
    + 23   # model features
    + 2    # target
)

assert len(final_model_dataset.columns) == expected_columns

print("\n" + "=" * 80)
print("FINAL DATASET SHAPE")
print("=" * 80)

print(
    f"Rows    : {len(final_model_dataset):,}"
)

print(
    f"Columns : {len(final_model_dataset.columns)}"
)

print("\nExpected:")
print("5 metadata + 23 model features + 2 target = 30 columns")

# ------------------------------------------------------------
# 10. DISPLAY FINAL TABLE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL 30-COLUMN DATASET — PREVIEW")
print("=" * 80)

display(
    final_model_dataset.head(10)
)

# ------------------------------------------------------------
# 11. DISPLAY COLUMN LIST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL COLUMN LIST")
print("=" * 80)

for i, col in enumerate(
    final_model_dataset.columns,
    1
):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# 12. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

target_summary = (
    final_model_dataset
    .groupby(
        ["target", "target_label"],
        sort=True
    )
    .size()
    .reset_index(
        name="observations"
    )
)

target_summary["percentage"] = (
    target_summary["observations"]
    / len(final_model_dataset)
    * 100
).round(2)

display(target_summary)

# ------------------------------------------------------------
# 13. CREATE X — ONLY 23 MODEL FEATURES
# ------------------------------------------------------------

X = final_model_dataset[
    model_features
].copy()

assert X.shape[1] == 23

# ------------------------------------------------------------
# 14. CREATE y — ONLY TARGET LABEL
# ------------------------------------------------------------

y = final_model_dataset[
    "target_label"
].copy()

# ------------------------------------------------------------
# 15. VERIFY X DOES NOT CONTAIN METADATA / FUTURE DATA
# ------------------------------------------------------------

assert "content_hash_id" not in X.columns
assert "client_hash_id" not in X.columns
assert "window_start" not in X.columns
assert "window_end" not in X.columns
assert "month" not in X.columns

assert not any(
    col.startswith("future_")
    for col in X.columns
)

# ------------------------------------------------------------
# 16. VERIFY Y
# ------------------------------------------------------------

assert y.name == "target_label"

assert y.isna().sum() == 0

assert set(
    y.unique()
).issubset(
    {"Down", "Flat", "Up"}
)

# ------------------------------------------------------------
# 17. SAVE FINAL DATASET AS PARQUET
# ------------------------------------------------------------

parquet_path = (
    "/content/final_model_dataset_30_columns.parquet"
)

final_model_dataset.to_parquet(
    parquet_path,
    index=False
)

# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL DATASET — COMPLETE")
print("=" * 80)

print(
    f"Final dataset rows    : "
    f"{len(final_model_dataset):,}"
)

print(
    f"Final dataset columns : "
    f"{final_model_dataset.shape[1]}"
)

print(
    f"X shape               : "
    f"{X.shape}"
)

print(
    f"y shape               : "
    f"{y.shape}"
)

print("\nDataset structure:")
print("  5 metadata columns")
print("  23 model features")
print("  1 encoded target")
print("  1 target label")

print("\n✓ Total = 30 columns")
print("✓ Future columns removed")
print("✓ Metadata retained")
print("✓ Month retained for filtering")
print("✓ X contains exactly 23 model features")
print("✓ y contains only target_label")
print("✓ Original df_final_ml remains unchanged")

print("\nParquet saved at:")
print(parquet_path)

W04 — REBUILD FINAL MODEL DATASET
Source rows: 2,705,303
Rows after duplicate cleanup: 2,705,303
Duplicates removed: 0

VALID 6-MONTH WINDOWS
Valid current → future windows: 978,801


KeyError: 'Required source features are missing:\nai_other_missing'

In [ ]:
df_final_ml.tail(20)

**Prepare Final Dataset for Model Training:**

In [ ]:
# ============================================================
# BLOCK 4.9 — FINAL MODEL DATASET
#
# FINAL DATASET = 30 COLUMNS
#
# 5 metadata columns
# + 23 model features
# + 2 target columns
#
# Metadata:
#   content_hash_id
#   client_hash_id
#   window_start
#   window_end
#   month
#
# Model features:
#   23 rolling/current features
#
# Target:
#   target       -> 0 / 1 / 2
#   target_label -> Down / Flat / Up
#
# NO future_* columns are retained.
# ============================================================

import pandas as pd

print("=" * 80)
print("W04 — FINAL MODEL DATASET")
print("=" * 80)

# ------------------------------------------------------------
# 1. EXACT 23 MODEL FEATURES
# ------------------------------------------------------------

model_features = [
    "gsc_clicks_mean_3m",
    "gsc_clicks_last",

    "gsc_impressions_mean_3m",
    "gsc_impressions_last",

    "gsc_avg_position_mean_3m",
    "gsc_avg_position_last",

    "ga4_total_engagement_sec_mean_3m",
    "ga4_total_engagement_sec_last",

    "sessions_organic_mean_3m",
    "sessions_organic_last",

    "sessions_ai_mean_3m",
    "sessions_ai_last",

    "gsc_avg_position_missing_mean_3m",
    "gsc_avg_position_missing_last",

    "ctr_mean_3m",
    "ctr_last",

    "sec_per_click_mean_3m",
    "sec_per_click_last",

    "ai_share_mean_3m",
    "ai_share_last",

    "engagement_per_organic_session_mean_3m",
    "engagement_per_organic_session_last",

    "current_imp_3m"
]

assert len(model_features) == 23

# ------------------------------------------------------------
# 2. METADATA COLUMNS
# ------------------------------------------------------------

metadata_columns = [
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end"
]

# ------------------------------------------------------------
# 3. CREATE MONTH COLUMN
#
# month = latest month of the current 3-month window
#
# Example:
# Jan + Feb + Mar
#       ↓
# month = Mar
# ------------------------------------------------------------

df_final_model = df_final_ml.copy()

df_final_model["month"] = pd.to_datetime(
    df_final_model["window_end"],
    errors="coerce"
)

# ------------------------------------------------------------
# 4. TARGET COLUMNS
# ------------------------------------------------------------

target_columns = [
    "target",
    "target_label"
]

# ------------------------------------------------------------
# 5. FINAL COLUMN ORDER
# ------------------------------------------------------------

final_columns = (
    metadata_columns
    + ["month"]
    + model_features
    + target_columns
)

# ------------------------------------------------------------
# 6. VERIFY ALL REQUIRED COLUMNS EXIST
# ------------------------------------------------------------

missing_required = [
    col
    for col in final_columns
    if col not in df_final_model.columns
]

if missing_required:

    raise KeyError(
        "Missing required columns:\n"
        + "\n".join(missing_required)
    )

# ------------------------------------------------------------
# 7. CREATE FINAL DATASET
# ------------------------------------------------------------

final_model_dataset = df_final_model[
    final_columns
].copy()

# ------------------------------------------------------------
# 8. FUTURE COLUMN CHECK
# ------------------------------------------------------------

future_columns = [
    col
    for col in final_model_dataset.columns
    if col.startswith("future_")
]

assert len(future_columns) == 0, (
    f"Future columns found: {future_columns}"
)

# ------------------------------------------------------------
# 9. FINAL SHAPE CHECK
# ------------------------------------------------------------

expected_columns = (
    5      # metadata
    + 23   # model features
    + 2    # target
)

assert len(final_model_dataset.columns) == expected_columns

print("\n" + "=" * 80)
print("FINAL DATASET SHAPE")
print("=" * 80)

print(
    f"Rows    : {len(final_model_dataset):,}"
)

print(
    f"Columns : {len(final_model_dataset.columns)}"
)

print("\nExpected:")
print("5 metadata + 23 model features + 2 target = 30 columns")

# ------------------------------------------------------------
# 10. DISPLAY FINAL TABLE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL 30-COLUMN DATASET — PREVIEW")
print("=" * 80)

display(
    final_model_dataset.head(10)
)

# ------------------------------------------------------------
# 11. DISPLAY COLUMN LIST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL COLUMN LIST")
print("=" * 80)

for i, col in enumerate(
    final_model_dataset.columns,
    1
):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# 12. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

target_summary = (
    final_model_dataset
    .groupby(
        ["target", "target_label"],
        sort=True
    )
    .size()
    .reset_index(
        name="observations"
    )
)

target_summary["percentage"] = (
    target_summary["observations"]
    / len(final_model_dataset)
    * 100
).round(2)

display(target_summary)

# ------------------------------------------------------------
# 13. CREATE X — ONLY 23 MODEL FEATURES
# ------------------------------------------------------------

X = final_model_dataset[
    model_features
].copy()

assert X.shape[1] == 23

# ------------------------------------------------------------
# 14. CREATE y — ONLY TARGET LABEL
# ------------------------------------------------------------

y = final_model_dataset[
    "target_label"
].copy()

# ------------------------------------------------------------
# 15. VERIFY X DOES NOT CONTAIN METADATA / FUTURE DATA
# ------------------------------------------------------------

assert "content_hash_id" not in X.columns
assert "client_hash_id" not in X.columns
assert "window_start" not in X.columns
assert "window_end" not in X.columns
assert "month" not in X.columns

assert not any(
    col.startswith("future_")
    for col in X.columns
)

# ------------------------------------------------------------
# 16. VERIFY Y
# ------------------------------------------------------------

assert y.name == "target_label"

assert y.isna().sum() == 0

assert set(
    y.unique()
).issubset(
    {"Down", "Flat", "Up"}
)

# ------------------------------------------------------------
# 17. SAVE FINAL DATASET AS PARQUET
# ------------------------------------------------------------

parquet_path = (
    "/content/final_model_dataset_30_columns.parquet"
)

final_model_dataset.to_parquet(
    parquet_path,
    index=False
)

# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL DATASET — COMPLETE")
print("=" * 80)

print(
    f"Final dataset rows    : "
    f"{len(final_model_dataset):,}"
)

print(
    f"Final dataset columns : "
    f"{final_model_dataset.shape[1]}"
)

print(
    f"X shape               : "
    f"{X.shape}"
)

print(
    f"y shape               : "
    f"{y.shape}"
)

print("\nDataset structure:")
print("  5 metadata columns")
print("  23 model features")
print("  1 encoded target")
print("  1 target label")

print("\n✓ Total = 30 columns")
print("✓ Future columns removed")
print("✓ Metadata retained")
print("✓ Month retained for filtering")
print("✓ X contains exactly 23 model features")
print("✓ y contains only target_label")
print("✓ Original df_final_ml remains unchanged")

print("\nParquet saved at:")
print(parquet_path)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method Choice and Why

### Strategy Overview
To address the SEO content decay task, we evaluate a combination of linear benchmarks, non-linear tree ensembles, and validation techniques. We select **Random Forest Classifier** as our primary champion model and **Logistic Regression** as our linear benchmark model.

---

### Comparison of Toolkit Methods

| Toolkit Method | Role in Workflow | Key Strengths | Why Included or Excluded for Decay Lane |
| :--- | :--- | :--- | :--- |
| **Correlation & Signal Analysis** | Feature Screening | Identifies multicollinearity and target leakage. | **Included (Pre-processing):** Essential to strictly remove forbidden trend variables (`trend_pct`, `trend_direction`). |
| **Grouped Validation** | Validation Design | Prevents data leakage across same-client pages. | **Included (Validation):** Grouping by `client_id` ensures the model generalizes to unseen domains rather than memorizing domain-specific baseline numbers. |
| **Logistic Regression** | Baseline ML Model | Simple, fast, and directly interpretable linear baseline. | **Included (ML Baseline):** Benchmark model to verify if a learned linear boundary beats our Week 4 rule-based baseline. |
| **Decision Tree** | Interpretable Model | Visualizable if-else logic trees. | **Included (Secondary):** Useful for quick rules extraction, though prone to higher variance on continuous traffic signals compared to ensembles. |
| **Random Forest** | **Primary Champion Model** | Ensemble of decision trees; handles non-linearities, outliers, and feature interactions. | **SELECTED CHAMPION:** Perfectly fits the power-law nature of web traffic and non-linear ranking drops. |
| **Gradient Boosting** | High-Capacity Model | Strong predictive power on structured tabular data. | **Tested with Constraints:** Evaluated cautiously with shallow depth to avoid overfitting noisy month-to-month traffic fluctuations. |
| **Permutation Importance** | Post-Hoc Interpretability | Measures true feature contribution by shuffling values post-training. | **Included (Sanity Check):** Verifies model honesty and guards against hidden proxy data leakage. |
| **Clustering (K-Means)** | Unsupervised Analysis | Segments content items into distinct performance tiers. | **Exploratory:** Used to analyze structural performance clusters prior to classification. |

---

### Why Random Forest Fits Our SEO Content Decay Lane

1. **Captures Non-Linear SEO Ranking Dynamics:**
   SEO ranking drops do not decay linearly. Losing Rank 1 to Rank 4 results in a catastrophic drop ($\approx 50\%+$) in impressions and CTR, whereas dropping from Rank 25 to Rank 28 has negligible impact. Random Forest handles these step-function thresholds naturally without requiring non-linear feature transformations.

2. **Robust to Heavy-Tailed Power-Law Distributions:**
   Search traffic (`gsc_impressions`) follows a steep power-law distribution where a small percentage of high-traffic pages dominate total volume. Random Forest uses threshold-based splits rather than distance metrics, making it scale-invariant and immune to extreme traffic outliers.

3. **Handles Multi-Signal Feature Interactions:**
   Content decay is rarely caused by a single metric. Random Forest automatically captures multi-variable interaction logic (e.g., *low impressions AND dropping position AND low engagement*) without requiring manual feature engineering.

4. **Transparent Feature Importance & Leakage Defense:**
   Combined with Permutation Importance, Random Forest provides clear insight into which features drive predictions. This ensures the model relies on true signals rather than memorizing forbidden trend indicators.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Validation & Split Design Strategy

### Strategy Overview
To evaluate our model under realistic production conditions, we implement a **Grouped, Time-Aware Split Strategy**.

This approach combines chronological partitioning with strict client-level grouping to systematically eliminate both temporal lookahead and domain identity leakage.

---

### Core Principles & Mechanics

1. **Primary Constraint — Time-Aware Split (Chronological Ordering):**
   * The dataset is ordered chronologically. Training strictly utilizes historical observation windows, while evaluation is performed on chronologically subsequent, unseen future windows.
   * **Justification:** Because the model leverages a past 3-month feature window to predict a future 3-month SEO outcome, future months must never enter the training set. This prevents future lookahead bias and macro-level temporal leakage (e.g., global search engine algorithm updates).

2. **Grouping Constraint — Client Isolation (`client_hash_id` Grouping):**
   * Client identity (`client_hash_id`) is preserved as a strict grouping boundary. All historical records for a specific client domain are isolated entirely within either the training or validation partition.
   * **Justification:** Prevents improper mixing of client-level data across evaluation sets. This ensures the model learns scale-invariant decay signals rather than memorizing domain-specific baselines.

---

### Production Alignment
This hybrid strategy directly mirrors FlyRank's real-world operational context: leveraging past observation windows to reliably predict future organic performance trajectory for existing and onboarding client domains.

In [ ]:
# ============================================================
# W05 — GROUPED + TIME-AWARE TRAIN / TEST SPLIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W05 — GROUPED + TIME-AWARE DATA SPLIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. COPY FINAL DATASET
# ------------------------------------------------------------

df_split = df_final_model.copy()

# Make sure month is datetime
df_split["month"] = pd.to_datetime(
    df_split["month"],
    errors="coerce"
)

# Remove invalid rows only if necessary
df_split = df_split.dropna(
    subset=["month", "client_hash_id", "target"]
).copy()

# Sort chronologically
df_split = df_split.sort_values(
    ["month", "client_hash_id"]
).reset_index(drop=True)

print(f"Total rows: {len(df_split):,}")

# ------------------------------------------------------------
# 2. TIME-AWARE SPLIT
#
# Earlier 80% of time -> TRAIN
# Latest 20% of time   -> TEST
# ------------------------------------------------------------

unique_months = (
    df_split["month"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

split_position = int(
    len(unique_months) * 0.80
)

train_months = unique_months.iloc[
    :split_position
]

test_months = unique_months.iloc[
    split_position:
]

train_end = train_months.max()
test_start = test_months.min()

# ------------------------------------------------------------
# 3. CREATE TRAIN / TEST
# ------------------------------------------------------------

train_df = df_split[
    df_split["month"] <= train_end
].copy()

test_df = df_split[
    df_split["month"] >= test_start
].copy()

# ------------------------------------------------------------
# 4. SAFETY CHECK
# ------------------------------------------------------------

assert train_df["month"].max() < test_df["month"].min()

# ------------------------------------------------------------
# 5. CLIENT OVERLAP CHECK
#
# Same clients may exist in both sets.
# This is intentional because we are predicting future
# performance of existing FlyRank clients.
# ------------------------------------------------------------

train_clients = set(
    train_df["client_hash_id"].unique()
)

test_clients = set(
    test_df["client_hash_id"].unique()
)

overlap_clients = train_clients.intersection(
    test_clients
)

# ------------------------------------------------------------
# 6. DISPLAY SPLIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN / TEST SPLIT")
print("=" * 80)

print(
    f"Train rows       : {len(train_df):,}"
)

print(
    f"Test rows        : {len(test_df):,}"
)

print(
    f"Train percentage : "
    f"{len(train_df) / len(df_split) * 100:.2f}%"
)

print(
    f"Test percentage  : "
    f"{len(test_df) / len(df_split) * 100:.2f}%"
)

print("\nTrain period:")
print(
    f"{train_df['month'].min().date()} "
    f"→ "
    f"{train_df['month'].max().date()}"
)

print("\nTest period:")
print(
    f"{test_df['month'].min().date()} "
    f"→ "
    f"{test_df['month'].max().date()}"
)

print("\nUnique clients:")
print(
    f"Train clients : {len(train_clients):,}"
)

print(
    f"Test clients  : {len(test_clients):,}"
)

print(
    f"Client overlap: {len(overlap_clients):,}"
)

# ------------------------------------------------------------
# 7. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 80)

display(
    train_df["target"]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

print("\n" + "=" * 80)
print("TEST TARGET DISTRIBUTION")
print("=" * 80)

display(
    test_df["target"]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

# ------------------------------------------------------------
# 8. FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SPLIT SAFETY CHECK")
print("=" * 80)

print(
    "✓ Time-aware split applied."
)

print(
    "✓ Training contains only earlier months."
)

print(
    "✓ Test contains only later months."
)

print(
    "✓ Client grouping information preserved."
)

print(
    "✓ No future test months are present in training."
)

print(
    "\nTRAINING DATASET:",
    train_df.shape
)

print(
    "TEST DATASET:",
    test_df.shape
)


In [ ]:
# ============================================================
# W05 — FINAL GROUPED + TIME-AWARE TRAIN / TEST SPLIT
#
# FINAL ML INPUT:
#   X_train = 23 features
#   X_test  = 23 features
#   y_train = target
#   y_test  = target
#
# Metadata is kept separately.
# Future columns are completely excluded.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W05 — FINAL GROUPED + TIME-AWARE TRAIN / TEST SPLIT")
print("=" * 80)


# ============================================================
# 1. FINAL MODEL DATASET
# ============================================================

df_split = df_final_model.copy()

df_split["month"] = pd.to_datetime(
    df_split["month"],
    errors="coerce"
)

df_split = df_split.dropna(
    subset=[
        "month",
        "client_hash_id",
        "target"
    ]
).copy()

# Sort chronologically
df_split = (
    df_split
    .sort_values(
        ["month", "client_hash_id"]
    )
    .reset_index(drop=True)
)

print(f"Total rows available: {len(df_split):,}")


# ============================================================
# 2. DEFINE EXACT 23 MODEL FEATURES
# ============================================================

feature_columns = [
    "gsc_clicks_mean_3m",
    "gsc_clicks_last",

    "gsc_impressions_mean_3m",
    "gsc_impressions_last",

    "gsc_avg_position_mean_3m",
    "gsc_avg_position_last",

    "ga4_total_engagement_sec_mean_3m",
    "ga4_total_engagement_sec_last",

    "sessions_organic_mean_3m",
    "sessions_organic_last",

    "sessions_ai_mean_3m",
    "sessions_ai_last",

    "gsc_avg_position_missing_mean_3m",
    "gsc_avg_position_missing_last",

    "ctr_mean_3m",
    "ctr_last",

    "sec_per_click_mean_3m",
    "sec_per_click_last",

    "ai_share_mean_3m",
    "ai_share_last",

    "engagement_per_organic_session_mean_3m",
    "engagement_per_organic_session_last",

    "current_imp_3m"
]


# ============================================================
# 3. VERIFY ALL 23 FEATURES EXIST
# ============================================================

missing_features = [
    col
    for col in feature_columns
    if col not in df_split.columns
]

if missing_features:

    raise ValueError(
        "Missing model features:\n"
        + "\n".join(missing_features)
    )

print("\n✓ All 23 model features are present.")


# ============================================================
# 4. VERIFY TARGET
# ============================================================

if "target" not in df_split.columns:

    raise ValueError(
        "Target column 'target' is missing."
    )

print("✓ Target column is present.")


# ============================================================
# 5. FUTURE / LEAKAGE CHECK
# ============================================================

future_columns = [
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct"
]

future_present = [
    col
    for col in future_columns
    if col in feature_columns
]

if future_present:

    raise ValueError(
        "FUTURE LEAKAGE FOUND IN MODEL FEATURES: "
        + str(future_present)
    )

print("✓ No future columns are included in model features.")


# ============================================================
# 6. TIME-AWARE SPLIT
#
# Latest 3 months = TEST
# Everything before that = TRAIN
#
# We do NOT randomly split rows.
# ============================================================

unique_months = (
    df_split["month"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

if len(unique_months) < 4:

    raise ValueError(
        "Not enough unique months for a 3-month test period."
    )

test_months = unique_months.iloc[-3:]

train_months = unique_months.iloc[:-3]

train_end = train_months.max()
test_start = test_months.min()


# ============================================================
# 7. CREATE TRAIN / TEST DATA
# ============================================================

train_df = df_split[
    df_split["month"].isin(train_months)
].copy()

test_df = df_split[
    df_split["month"].isin(test_months)
].copy()


# ============================================================
# 8. TIME SAFETY CHECK
# ============================================================

assert train_df["month"].max() < test_df["month"].min()

print("\n" + "=" * 80)
print("TIME-AWARE SPLIT")
print("=" * 80)

print(
    f"Training period : "
    f"{train_df['month'].min().date()} "
    f"→ "
    f"{train_df['month'].max().date()}"
)

print(
    f"Testing period  : "
    f"{test_df['month'].min().date()} "
    f"→ "
    f"{test_df['month'].max().date()}"
)

print(
    f"\nTraining rows : {len(train_df):,}"
)

print(
    f"Testing rows  : {len(test_df):,}"
)

print(
    f"Training %    : "
    f"{len(train_df) / len(df_split) * 100:.2f}%"
)

print(
    f"Testing %     : "
    f"{len(test_df) / len(df_split) * 100:.2f}%"
)


# ============================================================
# 9. CLIENT GROUPING CHECK
#
# Same client can legitimately appear in both train and test
# because we are predicting FUTURE performance of existing
# clients.
# ============================================================

train_clients = set(
    train_df["client_hash_id"].unique()
)

test_clients = set(
    test_df["client_hash_id"].unique()
)

client_overlap = (
    train_clients
    .intersection(test_clients)
)

print("\n" + "=" * 80)
print("CLIENT GROUPING CHECK")
print("=" * 80)

print(
    f"Training clients : {len(train_clients):,}"
)

print(
    f"Testing clients  : {len(test_clients):,}"
)

print(
    f"Clients appearing in both periods : "
    f"{len(client_overlap):,}"
)

print(
    "\n✓ Client IDs are preserved for grouping."
)

print(
    "✓ Same client may appear in train and test "
    "because test represents that client's future."
)


# ============================================================
# 10. CREATE X AND Y
# ============================================================

X_train = train_df[
    feature_columns
].copy()

X_test = test_df[
    feature_columns
].copy()

y_train = train_df[
    "target"
].copy()

y_test = test_df[
    "target"
].copy()


# ============================================================
# 11. FINAL FEATURE CHECK
# ============================================================

assert X_train.shape[1] == 23
assert X_test.shape[1] == 23

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert list(X_train.columns) == feature_columns
assert list(X_test.columns) == feature_columns


# ============================================================
# 12. ENSURE NO FUTURE COLUMNS
# ============================================================

for col in X_train.columns:

    if (
        "future" in col.lower()
        or "target" in col.lower()
    ):

        raise ValueError(
            f"Invalid model feature detected: {col}"
        )


# ============================================================
# 13. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 80)

train_target_summary = (
    y_train
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

train_target_summary["percentage"] = (
    train_target_summary["count"]
    / len(y_train)
    * 100
).round(2)

display(train_target_summary)


print("\n" + "=" * 80)
print("TEST TARGET DISTRIBUTION")
print("=" * 80)

test_target_summary = (
    y_test
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

test_target_summary["percentage"] = (
    test_target_summary["count"]
    / len(y_test)
    * 100
).round(2)

display(test_target_summary)


# ============================================================
# 14. FINAL ML DATASET SHAPES
# ============================================================

print("\n" + "=" * 80)
print("FINAL ML DATASET")
print("=" * 80)

print(
    f"X_train : {X_train.shape}"
)

print(
    f"X_test  : {X_test.shape}"
)

print(
    f"y_train : {y_train.shape}"
)

print(
    f"y_test  : {y_test.shape}"
)


# ============================================================
# 15. FINAL FEATURE LIST
# ============================================================

print("\n" + "=" * 80)
print("23 MODEL FEATURES")
print("=" * 80)

for i, col in enumerate(
    feature_columns,
    start=1
):

    print(
        f"{i:2}. {col}"
    )


# ============================================================
# 16. FINAL PREVIEW
# ============================================================

print("\n" + "=" * 80)
print("X_TRAIN — FIRST 5 ROWS")
print("=" * 80)

display(
    X_train.head()
)

print("\n" + "=" * 80)
print("Y_TRAIN — FIRST 5 VALUES")
print("=" * 80)

display(
    y_train.head()
)


# ============================================================
# 17. FINAL SAFETY VERDICT
# ============================================================

print("\n" + "=" * 80)
print("FINAL SPLIT VERDICT")
print("=" * 80)

print("✓ Time-aware split used.")
print("✓ Latest 3 months reserved for testing.")
print("✓ Older months used for training.")
print("✓ Client ID preserved for grouping.")
print("✓ Same client may appear across time periods intentionally.")
print("✓ Exactly 23 model features.")
print("✓ Target kept separately.")
print("✓ Future target-generation columns excluded.")
print("✓ No random row-level split used.")

print("\nReady for model training.")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Dataset Standard: Same X_test and y_test

Models & Baselines:

Early Drop baseline

Logistic Regression

Random Forest

Metrics & Analysis:

Precision / Recall / F1

True Decay catch rate

Rank correlation with low-impression baseline

Action-threshold comparison

Confusion matrices

In [ ]:
# ============================================================
# W05 — FINAL MODEL vs BASELINE COMPARISON
#
# Target encoding:
#   0 = Down
#   1 = Flat
#   2 = Up
#
# Models:
#   1. Logistic Regression
#   2. Random Forest
#
# Early Drop baseline:
#   gsc_impressions_last < gsc_impressions_mean_3m
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from scipy.stats import spearmanr


print("=" * 80)
print("W05 — FINAL MODEL vs BASELINE COMPARISON")
print("=" * 80)


# ============================================================
# 1. TARGET ENCODING VERIFICATION
# ============================================================

print("\n" + "=" * 80)
print("TARGET ENCODING VERIFICATION")
print("=" * 80)

TARGET_MAP = {
    0: "Down",
    1: "Flat",
    2: "Up"
}

unique_targets = sorted(
    pd.Series(y_test).dropna().unique().tolist()
)

print("Unique target values:", unique_targets)

assert set(unique_targets).issubset(
    {0, 1, 2}
), "Unexpected target values found."

print("\nConfirmed target mapping:")

for value, label in TARGET_MAP.items():
    print(f"{value} = {label}")

print("\n✓ Target encoding confirmed.")


# ============================================================
# 2. INPUT CHECK
# ============================================================

print("\n" + "=" * 80)
print("MODEL INPUT CHECK")
print("=" * 80)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

assert X_train.shape[1] == 23
assert X_test.shape[1] == 23

print("\n✓ Exactly 23 model features.")


# ============================================================
# 3. TRAIN LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 80)
print("TRAINING — LOGISTIC REGRESSION")
print("=" * 80)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(
    X_train,
    y_train
)

print("✓ Logistic Regression trained.")


# ============================================================
# 4. RANDOM FOREST CHECK
# ============================================================

print("\n" + "=" * 80)
print("RANDOM FOREST")
print("=" * 80)

# If rf_model already exists from the training block,
# use it. Otherwise train it here.

if "rf_model" not in globals():

    from sklearn.ensemble import RandomForestClassifier

    print("Random Forest not found.")
    print("Training Random Forest now...")

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    rf_model.fit(
        X_train,
        y_train
    )

    print("✓ Random Forest trained.")

else:

    print("✓ Existing Random Forest model found.")


# ============================================================
# 5. PREDICTIONS
# ============================================================

print("\n" + "=" * 80)
print("GENERATING TEST PREDICTIONS")
print("=" * 80)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)

# IMPORTANT:
# class 0 = DOWN

lr_down_index = list(
    lr_model.classes_
).index(0)

rf_down_index = list(
    rf_model.classes_
).index(0)

lr_down_probability = lr_prob[
    :, lr_down_index
]

rf_down_probability = rf_prob[
    :, rf_down_index
]

print("✓ Logistic Regression predictions generated.")
print("✓ Random Forest predictions generated.")
print("✓ DOWN probability extracted.")


# ============================================================
# 6. MODEL METRICS
# ============================================================

def model_metrics(
    name,
    y_true,
    y_pred
):

    return {
        "model": name,

        "accuracy": (
            accuracy_score(
                y_true,
                y_pred
            ) * 100
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred
            ) * 100
        ),

        "macro_precision": (
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ) * 100
        ),

        "macro_recall": (
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ) * 100
        ),

        "macro_f1": (
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ) * 100
        ),

        "DOWN_precision": (
            precision_score(
                y_true,
                y_pred,
                labels=[0],
                average="macro",
                zero_division=0
            ) * 100
        ),

        "DOWN_recall": (
            recall_score(
                y_true,
                y_pred,
                labels=[0],
                average="macro",
                zero_division=0
            ) * 100
        ),

        "DOWN_f1": (
            f1_score(
                y_true,
                y_pred,
                labels=[0],
                average="macro",
                zero_division=0
            ) * 100
        )
    }


model_comparison = pd.DataFrame([
    model_metrics(
        "Logistic Regression",
        y_test,
        lr_pred
    ),

    model_metrics(
        "Random Forest",
        y_test,
        rf_pred
    )
])

model_comparison = model_comparison.round(2)

print("\n" + "=" * 80)
print("MODEL PERFORMANCE")
print("=" * 80)

display(model_comparison)


# ============================================================
# 7. CLASSIFICATION REPORTS
# ============================================================

print("\n" + "=" * 80)
print("RANDOM FOREST — CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        rf_pred,
        labels=[0, 1, 2],
        target_names=[
            "DOWN",
            "FLAT",
            "UP"
        ],
        zero_division=0
    )
)


print("\n" + "=" * 80)
print("LOGISTIC REGRESSION — CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        lr_pred,
        labels=[0, 1, 2],
        target_names=[
            "DOWN",
            "FLAT",
            "UP"
        ],
        zero_division=0
    )
)


# ============================================================
# 8. CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 80)
print("RANDOM FOREST — CONFUSION MATRIX")
print("=" * 80)

rf_cm = pd.DataFrame(
    confusion_matrix(
        y_test,
        rf_pred,
        labels=[0, 1, 2]
    ),
    index=[
        "Actual DOWN",
        "Actual FLAT",
        "Actual UP"
    ],
    columns=[
        "Pred DOWN",
        "Pred FLAT",
        "Pred UP"
    ]
)

display(rf_cm)


print("\n" + "=" * 80)
print("LOGISTIC REGRESSION — CONFUSION MATRIX")
print("=" * 80)

lr_cm = pd.DataFrame(
    confusion_matrix(
        y_test,
        lr_pred,
        labels=[0, 1, 2]
    ),
    index=[
        "Actual DOWN",
        "Actual FLAT",
        "Actual UP"
    ],
    columns=[
        "Pred DOWN",
        "Pred FLAT",
        "Pred UP"
    ]
)

display(lr_cm)


# ============================================================
# 9. EARLY DROP BASELINE
#
# Current standardized signal:
#
# latest impressions < current 3-month average
# ============================================================

print("\n" + "=" * 80)
print("BASELINE — EARLY DROP SIGNAL")
print("=" * 80)

required_columns = [
    "gsc_impressions_last",
    "gsc_impressions_mean_3m"
]

missing_columns = [
    col
    for col in required_columns
    if col not in X_test.columns
]

if missing_columns:

    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

early_drop_mask = (
    X_test["gsc_impressions_last"].to_numpy()
    <
    X_test["gsc_impressions_mean_3m"].to_numpy()
)

true_down = (
    np.asarray(y_test) == 0
)

early_drop_precision = (
    precision_score(
        true_down,
        early_drop_mask,
        zero_division=0
    ) * 100
)

early_drop_recall = (
    recall_score(
        true_down,
        early_drop_mask,
        zero_division=0
    ) * 100
)

early_drop_f1 = (
    f1_score(
        true_down,
        early_drop_mask,
        zero_division=0
    ) * 100
)

print(
    f"Early Drop signals : "
    f"{early_drop_mask.sum():,}"
)

print(
    f"Actual DOWN pages  : "
    f"{true_down.sum():,}"
)

print(
    f"Precision : {early_drop_precision:.2f}%"
)

print(
    f"Recall    : {early_drop_recall:.2f}%"
)

print(
    f"F1-score  : {early_drop_f1:.2f}%"
)


# ============================================================
# 10. LOW-IMPRESSION BASELINE SCORE
# ============================================================

print("\n" + "=" * 80)
print("BASELINE — LOW IMPRESSION PRIORITY SCORE")
print("=" * 80)

current_impressions = pd.to_numeric(
    X_test["gsc_impressions_last"],
    errors="coerce"
).fillna(0).clip(lower=0)

log_imp = np.log1p(
    current_impressions.to_numpy()
)

max_log = log_imp.max()

if max_log > 0:

    baseline_priority_score = (
        1 - (log_imp / max_log)
    )

else:

    baseline_priority_score = np.zeros(
        len(X_test)
    )

baseline_priority_score = np.clip(
    baseline_priority_score,
    0,
    1
)

print("✓ Low-impression priority score created.")


# ============================================================
# 11. MODEL PRIORITY SCORE
#
# Higher DOWN probability = higher priority
# ============================================================

lr_priority_score = lr_down_probability
rf_priority_score = rf_down_probability


# ============================================================
# 12. SPEARMAN RANK CORRELATION
# ============================================================

print("\n" + "=" * 80)
print("RANK CORRELATION")
print("=" * 80)

lr_spearman, lr_p = spearmanr(
    lr_priority_score,
    baseline_priority_score
)

rf_spearman, rf_p = spearmanr(
    rf_priority_score,
    baseline_priority_score
)

rank_table = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "spearman_r": [
        lr_spearman,
        rf_spearman
    ],

    "p_value": [
        lr_p,
        rf_p
    ]
}).round(4)

display(rank_table)


# ============================================================
# 13. TOP 10% PRIORITY OVERLAP
# ============================================================

def top_k_overlap(
    model_score,
    baseline_score,
    fraction=0.10
):

    n = len(model_score)

    k = max(
        1,
        int(n * fraction)
    )

    model_top = set(
        np.argsort(
            -model_score
        )[:k]
    )

    baseline_top = set(
        np.argsort(
            -baseline_score
        )[:k]
    )

    overlap = len(
        model_top.intersection(
            baseline_top
        )
    )

    precision_at_k = (
        overlap / k * 100
    )

    return k, overlap, precision_at_k


lr_k, lr_overlap, lr_p10 = top_k_overlap(
    lr_priority_score,
    baseline_priority_score
)

rf_k, rf_overlap, rf_p10 = top_k_overlap(
    rf_priority_score,
    baseline_priority_score
)

top_k_table = pd.DataFrame({

    "model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "K_top_10pct": [
        lr_k,
        rf_k
    ],

    "overlap": [
        lr_overlap,
        rf_overlap
    ],

    "precision_at_K_pct": [
        lr_p10,
        rf_p10
    ]
}).round(2)

print("\n" + "=" * 80)
print("TOP 10% PRIORITY AGREEMENT")
print("=" * 80)

display(top_k_table)


# ============================================================
# 14. ACTION LABEL THRESHOLDS
#
# >= 0.90 = URGENT_REFRESH
# >= 0.75 = HIGH_PRIORITY
# >= 0.50 = REVIEW
# <  0.50 = MONITOR
# ============================================================

def action_label(score):

    return np.select(
        [
            score >= 0.90,
            score >= 0.75,
            score >= 0.50
        ],
        [
            "URGENT_REFRESH",
            "HIGH_PRIORITY",
            "REVIEW"
        ],
        default="MONITOR"
    )


rf_action = action_label(
    rf_priority_score
)

lr_action = action_label(
    lr_priority_score
)


# ============================================================
# 15. ACTIONABLE DOWN METRICS
# ============================================================

rf_actionable = (
    rf_priority_score >= 0.50
)

lr_actionable = (
    lr_priority_score >= 0.50
)

action_metrics = pd.DataFrame({

    "method": [
        "Random Forest",
        "Logistic Regression",
        "Early Drop Baseline"
    ],

    "precision_pct": [
        precision_score(
            true_down,
            rf_actionable,
            zero_division=0
        ) * 100,

        precision_score(
            true_down,
            lr_actionable,
            zero_division=0
        ) * 100,

        early_drop_precision
    ],

    "recall_pct": [
        recall_score(
            true_down,
            rf_actionable,
            zero_division=0
        ) * 100,

        recall_score(
            true_down,
            lr_actionable,
            zero_division=0
        ) * 100,

        early_drop_recall
    ],

    "f1_pct": [
        f1_score(
            true_down,
            rf_actionable,
            zero_division=0
        ) * 100,

        f1_score(
            true_down,
            lr_actionable,
            zero_division=0
        ) * 100,

        early_drop_f1
    ]
}).round(2)

print("\n" + "=" * 80)
print("ACTION LABEL — DOWN DETECTION")
print("=" * 80)

display(action_metrics)


# ============================================================
# 16. FINAL COMPARISON
# ============================================================

final_comparison = pd.DataFrame({

    "method": [
        "Logistic Regression",
        "Random Forest",
        "Early Drop Baseline"
    ],

    "DOWN_precision_pct": [
        precision_score(
            true_down,
            lr_pred == 0,
            zero_division=0
        ) * 100,

        precision_score(
            true_down,
            rf_pred == 0,
            zero_division=0
        ) * 100,

        early_drop_precision
    ],

    "DOWN_recall_pct": [
        recall_score(
            true_down,
            lr_pred == 0,
            zero_division=0
        ) * 100,

        recall_score(
            true_down,
            rf_pred == 0,
            zero_division=0
        ) * 100,

        early_drop_recall
    ],

    "DOWN_f1_pct": [
        f1_score(
            true_down,
            lr_pred == 0,
            zero_division=0
        ) * 100,

        f1_score(
            true_down,
            rf_pred == 0,
            zero_division=0
        ) * 100,

        early_drop_f1
    ],

    "balanced_accuracy_pct": [
        balanced_accuracy_score(
            y_test,
            lr_pred
        ) * 100,

        balanced_accuracy_score(
            y_test,
            rf_pred
        ) * 100,

        np.nan
    ]
}).round(2)

print("\n" + "=" * 80)
print("FINAL MODEL vs BASELINE COMPARISON")
print("=" * 80)

display(final_comparison)


# ============================================================
# 17. FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("W05 — COMPARISON COMPLETE")
print("=" * 80)

print("""
Target:
    0 = DOWN
    1 = FLAT
    2 = UP

Frozen target definition:
    DOWN <= -30%
    FLAT  > -30% and < +50%
    UP   >= +50%

Early Drop:
    gsc_impressions_last < gsc_impressions_mean_3m

Model priority:
    P(DOWN)

Action thresholds:
    >= 0.90 = URGENT_REFRESH
    >= 0.75 = HIGH_PRIORITY
    >= 0.50 = REVIEW
    <  0.50 = MONITOR

✓ Same X_test used.
✓ Same y_test ground truth used.
✓ 23 model features only.
✓ No future target columns used as model inputs.
✓ Logistic Regression trained in this block.
✓ Random Forest trained/used.
✓ Early Drop baseline evaluated.
✓ Rank correlation evaluated.
✓ Top-10% priority agreement evaluated.
✓ Actionable DOWN detection evaluated.
""")

**Random Forest tuning code**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.